## assumptions

1. Please consider that my model is not enterprise, on test set get 85% of accuracy, some rare classes are a issue because original dataset have unbalance distribution and not enough data for hard classification task

 2. I'm developing a best-effort approach for a real-time microservice that performs drift detection filtering to be deploy into the 5G layer

## Drift detection

Using another heavy autoencoder model can not be ideal for limited HW scenario.

For this reason, I am trying to find out how to execute a less expensive drift detection operation before developing a microservice for the 5G k8s Layer.

I am interested into a portable and general solution.

---------------------------------------------------------------------------

> I want use a geometric approach.

----------------------------------------------------------------------------
This project has been devided around four main stages:
1. STAGE 1: Embedding extraction using a CNN model developed by me
2. STAGE 2: Visualize the embedding distribution and useful patterns
2. STAGE 3: Develop a robust dynamic thresholding policy to distinguish drifted data from clean/healthy data
2. STAGE 4: Test the thresholding policy using a sliding window scenario and gather performance metrics such as accuracy, precision, recall, and F1-score
3. STAGE 5: Use the OOD samples to retrain and tune the model
----------------------------------------------------------------------------

### I am interested in semantic drift

Discover how the model behaves by analyzing its embeddings when given:

- In-domain images: Grocery items with unusual or peculiar shapes.
- Out-of-domain images: Images completely outside the grocery domain.

Images that are very close to the original dataset are difficult to filter out because deep learning models generalize well. Forcing the model to change this behavior could lead to overfitting and a drop in performance.

Therefore, the focus is on detecting distinct semantic drift using images with unique and interesting features.


## STAGE 1

In [ ]:
!pip install image-classifiers
!git clone https://github.com/marcusklasson/GroceryStoreDataset.git # training dataset

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split
from huggingface_hub import hf_hub_download
import tensorflow as tf
import keras
from keras import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from classification_models.tfkeras import Classifiers
from google.colab import drive
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
import plotly.express as px
import seaborn as sns
from sklearn.neighbors import NearestNeighbors

from huggingface_hub import snapshot_download
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from collections import deque
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    silhouette_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, accuracy_score, precision_score,
    recall_score, f1_score
)


Grocery Store detection is my CNN model for testing.

Experiment conclusion are not related to the model but are general.

In [ ]:
REPO_ID = "tomasconti/Drift_Detection"
print("Download RestNet18 model")

config_path = hf_hub_download(repo_id=REPO_ID, filename="resnet18_v1/config.json")
weights_path = hf_hub_download(repo_id=REPO_ID, filename="resnet18_v1/model.weights.h5")

with open(config_path, "r") as f:
    model_config = json.load(f)

model = keras.saving.deserialize_keras_object(model_config)
model.load_weights(weights_path)
print("\nRestNet18 loaded")
model.summary()

ResNet18, preprocess_input = Classifiers.get('resnet18') # need of preprocess_input for the ImageDataGenerator, creating streaming sample batch
embedding_extractor = Model(inputs=model.input, outputs=model.get_layer("embedding_layer").output)


In [ ]:
ROOT_DIR = "/content/GroceryStoreDataset/dataset/"
def process_dataframe(df, root_dir):
    df['path'] = df['path'].apply(lambda x: os.path.join(root_dir, x))
    df['fine_label'] = df['fine_label'].astype(str)
    return df

train_df = pd.read_csv(os.path.join(ROOT_DIR, "train.txt"), header=None, sep=",", names=['path', 'fine_label', 'coarse_label'])
val_df = pd.read_csv(os.path.join(ROOT_DIR, "val.txt"), header=None, sep=",",names=['path', 'fine_label', 'coarse_label'])
test_df = pd.read_csv(os.path.join(ROOT_DIR, "test.txt"), header=None, sep=",", names=['path', 'fine_label', 'coarse_label'])

for df in (train_df, val_df, test_df):
    df.drop(columns=["coarse_label"], inplace=True)

train_df = process_dataframe(train_df, ROOT_DIR)
val_df = process_dataframe(val_df, ROOT_DIR)
test_df = process_dataframe(test_df, ROOT_DIR)

combined_df = pd.concat([train_df, val_df], ignore_index=True) #This operation has been explained into the cnn model dev. I need beacause valset does not have all the classes

train_new, val_new = train_test_split(
    combined_df,
    test_size=0.20,
    stratify=combined_df['fine_label'],
    random_state=42
)

print(f"Train: {len(train_new)} | Val: {len(val_new)} | Test: {len(test_df)}")
print(f"Number of fine-grained classes: {combined_df['fine_label'].nunique()}\n")

print( len(train_df['fine_label'].unique()))
print(train_df['fine_label'].unique())

print( len(val_df['fine_label'].unique()))
print(val_df['fine_label'].unique())

print( len(test_df['fine_label'].unique()))
print(test_df['fine_label'].unique())

print( len(train_new['fine_label'].unique()))
print( len(val_new['fine_label'].unique()))
print( len(test_df['fine_label'].unique()))

train_new.head(5)


In [ ]:
def create_generator_from_dataframe(df, batch_size=32):
    datagen = ImageDataGenerator(preprocessing_function=preprocess_input) #ideal to define data streaming flow to get into the models
    # 256 x256 becasue the model first layer already resize the imagine into 224 an 224
    #lazily-loaded data stream
    return datagen.flow_from_dataframe( dataframe=df, x_col='path', y_col='fine_label', target_size=(256, 256), batch_size=64, class_mode='categorical', shuffle=False, validate_filenames=False )
train_gen = create_generator_from_dataframe(train_new)
val_gen   = create_generator_from_dataframe(val_new)
test_gen  = create_generator_from_dataframe(test_df)

If you want to load the dataset from the local directory use this cell

In [ ]:

"""
YOGURT_PATH = "/content/drive/MyDrive/Yogurt "
STILL_WATER_PATH = "/content/drive/MyDrive/Still_water_images"
def plot_random_images_inline(folder_path, title_prefix):

    files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg'))] # extract all the valids file name into the dir path

    num_to_sample = min(3, len(files)) #peack random 3
    random_files = random.sample(files, num_to_sample)

    fig, axes = plt.subplots(1, num_to_sample, figsize=(15, 5))
    for i, img_name in enumerate(random_files):
        img = Image.open(os.path.join(folder_path, img_name)) #open the random file name images and plot it
        img = ImageOps.exif_transpose(img)

        axes[i].imshow(img)
        axes[i].set_title(f"{title_prefix}: {img_name}", fontsize=10)
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

plot_random_images_inline(YOGURT_PATH, "Yogurt")
plot_random_images_inline(STILL_WATER_PATH, "Still water")
"""
"""
#preprocess_input -----> allow the preprocessing operation needed for restnest 18 elaboration (example standard scaling ...)
def create_generator_from_dataframe(df, batch_size=32):
    datagen = ImageDataGenerator(preprocessing_function=preprocess_input) #ideal to define data streaming flow to get into the models
    # 256 x256 becasue the model first layer already resize the imagine into 224 an 224
    return datagen.flow_from_dataframe( dataframe=df, x_col='path', y_col='fine_label', target_size=(256, 256), batch_size=batch_size, class_mode='categorical', shuffle=False, validate_filenames=False )

def create_generator_from_folder(folder_path, batch_size=32):
    image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg'))]

    df = pd.DataFrame({'path': [os.path.join(folder_path, f) for f in image_files]}) #extract the absolute path of images

    datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

    return datagen.flow_from_dataframe( dataframe=df, x_col='path', y_col=None, target_size=(256, 256), batch_size=batch_size, class_mode=None, shuffle=False, validate_filenames=False )

train_gen = create_generator_from_dataframe(train_new)
val_gen   = create_generator_from_dataframe(val_new)
test_gen  = create_generator_from_dataframe(test_df)

yogurt_gen = create_generator_from_folder(YOGURT_PATH)
water_gen  = create_generator_from_folder(STILL_WATER_PATH)
"""

Loading the project dataset from my hugging face repository

In [ ]:
#laod the hugging face repo into LOCAL_REPO
#/root/.cache/huggingface/hub/models--tomasconti--Drift_Detection/snapshots/1ec8152c78669....
REPO_ID = "tomasconti/Drift_Detection"
LOCAL_REPO = snapshot_download( repo_id=REPO_ID, repo_type="model" )
print("Repo loaded into :", LOCAL_REPO)

def get_local_files(folder_keyword):
    image_files = []
    keyword = folder_keyword.lower()

    for root, _, filenames in os.walk(LOCAL_REPO): #recursion looking for the dir with all that that start by keyward
        if keyword in root.lower():
            #print(f"\n root fine out ->  {root.lower()}")
            #print(f"\n file system fine out ->  {filenames}")
            for filename in filenames: #for each file
                if filename.lower().endswith((".png", ".jpg")): #check if it is a images
                    image_files.append(os.path.join(root, filename)) #laod it

    return sorted(image_files)


def create_generator_from_repo_v2(folder_name, batch_size=32):
    file_paths = get_local_files(folder_name) #go into the folder

    if not file_paths:
        raise ValueError(
            f" ERROR no images'{folder_name}'."
        )

    print(f"\n {folder_name}: with inside {len(file_paths)} imagies")

    df = pd.DataFrame({ "path": file_paths  })

    datagen = ImageDataGenerator( preprocessing_function=preprocess_input )

    return datagen.flow_from_dataframe(
        dataframe=df,
        x_col="path",
        y_col=None,
        target_size=(256, 256),
        batch_size=64,
        class_mode=None,
        shuffle=False,
        validate_filenames=False
    )


for cls in ["Still_water_images", "Yogurt", "Eags"]:
    print(cls, len(get_local_files(cls)))
#return the DirectoryIterator class for each set, lazily-loaded datastream
water_gen = create_generator_from_repo_v2("Still_water_images")
yogurt_gen = create_generator_from_repo_v2("Yogurt")
eags_gen = create_generator_from_repo_v2("Eags")


Is important to consider different position and setting of items

The eggs cover sometimes is close (very important)

In [ ]:
def plot_grid_from_generators(generators_dict, num_samples):

    categories = list(generators_dict.keys())
    n_cats = len(categories)
    fig, axes = plt.subplots(n_cats, num_samples, figsize=(num_samples * 4, n_cats * 4))

    if n_cats == 1 and num_samples == 1: axes = [[axes]]
    elif n_cats == 1: axes = [axes]
    elif num_samples == 1: axes = [[ax] for ax in axes]

    for row, cat in enumerate(categories):
        gen = generators_dict[cat]

        df_files = gen.filenames


        actual_samples = min(num_samples, len(df_files))
        sample_files = random.sample(df_files, actual_samples)

        for col, file_path in enumerate(sample_files):
            img = Image.open(file_path)
            img = ImageOps.exif_transpose(img)

            axes[row][col].imshow(img)
            axes[row][col].set_title(f"{cat}", fontsize=12)
            axes[row][col].axis('off')

        for col in range(actual_samples, num_samples):
            axes[row][col].axis('off')

    plt.tight_layout()
    plt.show()
#dictionary
my_generators = {  "Still Water": water_gen, "Yogurt": yogurt_gen, "Eags": eags_gen }
plot_grid_from_generators(my_generators, num_samples=10)

In [ ]:
my_generators2 = {  "Train": train_gen}
plot_grid_from_generators(my_generators2, num_samples=10)

After loaded all images extract embeddings from the model's final layer.

Embeddings are vector representations of items in the model's latent feature space.

In [ ]:
def extract_and_save(generator, name):
    generator.reset()
    embeddings = embedding_extractor.predict(generator, verbose=1) #model get in is a image streaming
    np.save(f"/content/{name}_embeddings.npy", embeddings)
    print(f"Saved {name}_embeddings.npy and hape: {embeddings.shape}")
#lazily-loaded stream as input to models
extract_and_save(train_gen, "grocery_train")
extract_and_save(val_gen,   "grocery_val")
extract_and_save(test_gen,  "grocery_test")
extract_and_save(yogurt_gen, "yogurt")
extract_and_save(water_gen,  "still_water")

## STAGE 2
Load the dataset from the hugging face repo.


In [ ]:
REPO_ID = "tomasconti/Drift_Detection"
print("Downloading native 512-D embeddings from HuggingFace")

path_train = hf_hub_download(repo_id=REPO_ID, filename="good_to_delivery/grocery_train_embeddings.npy")

path_yogurt = hf_hub_download(repo_id=REPO_ID, filename="good_to_delivery/yogurt_embeddings.npy")

path_test = hf_hub_download(repo_id=REPO_ID, filename="good_to_delivery/grocery_test_embeddings.npy")
path_val = hf_hub_download(repo_id=REPO_ID, filename="good_to_delivery/grocery_val_embeddings.npy")
path_water = hf_hub_download(repo_id=REPO_ID, filename="good_to_delivery/still_water_embeddings.npy")
path_egg = hf_hub_download(repo_id=REPO_ID, filename="good_to_delivery/egg_embeddings.npy")

test_embeddings = np.load(path_test)
val_embeddings = np.load(path_val)
train_embeddings = np.load(path_train)
embeddings_water = np.load(path_water)
yogurt_embeddings = np.load(path_yogurt)
egg_embeddings = np.load(path_egg)

In [ ]:
def explore_embeddings(data, label):
    print(f"\n--- Analysis: {label} ---")
    print(f"Shape: {data.shape}")
    print(f"Mean value: {np.mean(data):.4f}")
    print(f"Standard Deviation: {np.std(data):.4f}")
    print(f"Range: [{np.min(data):.4f}, {np.max(data):.4f}]")

    # Are there any NaNs or Infinities? cloud be a problem in case
    has_nan = np.isnan(data).any()
    has_inf = np.isinf(data).any()
    print(f"Contains NaN: {has_nan} | Contains Inf: {has_inf}")

explore_embeddings(train_embeddings, "Grocery Train")
explore_embeddings(yogurt_embeddings, "Yogurt")
explore_embeddings(test_embeddings, "Grocery Test")
explore_embeddings(val_embeddings, "val ")
explore_embeddings(train_embeddings, "train")
explore_embeddings(egg_embeddings, "egg")

In [ ]:
data = { #dictionary
    "Train": train_embeddings,
    "Val": val_embeddings,
    "Test": test_embeddings,
    "Yogurt": yogurt_embeddings,
    "Water": embeddings_water,
    "Egg": egg_embeddings
}

In [ ]:

#for each of the set of embedding creare a hist to show up the values distributions

plt.figure(figsize=(12, 7))
for label, embeddings in data.items(): #over all the dataset of embeddings
    plt.hist(embeddings.flatten(), bins=50, alpha=0.3, label=label, density=True)

plt.legend()
plt.title("Embedding Distribution Comparison - All Datasets")
plt.xlabel("Feature Value")
plt.ylabel("Density")
plt.grid(axis='y', alpha=0.3)
plt.show()

fig, axes = plt.subplots(3, 2, figsize=(15, 12))
axes = axes.flatten()

#do not consider all the dataset but one set at time
for i, (label, embeddings) in enumerate(data.items()): #one dataset after the other
    axes[i].hist(embeddings.flatten(), bins=50, color='skyblue', alpha=0.7, density=True)
    axes[i].set_title(f"Distribution: {label}")
    axes[i].set_xlabel("Feature Value")
    axes[i].set_ylabel("Density")
    axes[i].grid(axis='y', alpha=0.3)
#axes[5].axis('off') #empty

plt.tight_layout()
plt.show()

### Embedding Distribution Analysis
1. Sparse Representation:
    The spike at 0 is primarily due to ReLU deactivations
2. Domain Alignment:
    The overlapping histograms confirm the model uses a
    consistent feature space Embedding differences are minimals, means the network try to extract the same features trough the model layers but some activation are differents

Now try to plot the distribution into a 2D space but also into a 3D optional space, try to see if there are some usefull pattern

In [ ]:
datasets = ["Train", "Val", "Test"] #visualize alone

combined_data = np.concatenate([data["Test"], data["Water"], data["Yogurt"], data["Egg"]]) #create big dataset combine
#creation of labels
combined_labels = ["Test"]*len(data["Test"]) + ["Water"]*len(data["Water"]) + ["Yogurt"]*len(data["Yogurt"]) +["Egg"]*len(data["Egg"])

def run_tsne(embeddings, n_dim):
    #perplexity -> check more neighbornhood, more stability | pca -> algorithm to reduce the dim
    return TSNE(n_components=n_dim, perplexity=50, random_state=42, init="pca").fit_transform(embeddings)

projs_2d = { name: run_tsne(data[name], 2) for name in datasets} # exec tsne on each dataset
projs_2d["Combined"] = run_tsne(combined_data, 2) #exec on all the dataset combine

projs_3d = {name: run_tsne(data[name], 3) for name in datasets}
projs_3d["Combined"] = run_tsne(combined_data, 3)
#################################################################################################
# 2D PLOT
fig_2d, axes_2d = plt.subplots(2, 2, figsize=(14, 12))
axes_2d = axes_2d.flatten()
#single 2d plot
for i, name in enumerate(datasets):
    axes_2d[i].scatter(projs_2d[name][:, 0], projs_2d[name][:, 1], alpha=0.6, s=20)
    axes_2d[i].set_title(f"{name} Set (2D)")
    axes_2d[i].grid(alpha=0.3)
#combine 2d plot
for label in np.unique(combined_labels):
    mask = (np.array(combined_labels) == label)
    axes_2d[3].scatter(projs_2d["Combined"][mask, 0], projs_2d["Combined"][mask, 1], label=label, alpha=0.6, s=30)
axes_2d[3].legend()
axes_2d[3].set_title("Test + Water + Yogurt (2D)")
plt.tight_layout()
plt.show()

# # OPTIONAL THE 3D PLOT
fig_3d = plt.figure(figsize=(14, 12))
for i, name in enumerate(datasets):
    ax = fig_3d.add_subplot(2, 2, i+1, projection='3d')
    ax.scatter(projs_3d[name][:, 0], projs_3d[name][:, 1], projs_3d[name][:, 2], alpha=0.6, s=20)
    ax.set_title(f"{name} Set (3D)")

ax_comb = fig_3d.add_subplot(2, 2, 4, projection='3d')
for label in np.unique(combined_labels):
    mask = (np.array(combined_labels) == label)
    ax_comb.scatter(projs_3d["Combined"][mask, 0], projs_3d["Combined"][mask, 1], projs_3d["Combined"][mask, 2], label=label, alpha=0.6, s=30)
ax_comb.legend()
ax_comb.set_title("Test + Water + Yogurt (3D Static)")
plt.tight_layout()
plt.show()

# OPTIONAL THE 3D DYNAMIC PLOT
df_3d = pd.DataFrame(projs_3d["Combined"], columns=['x', 'y', 'z'])
df_3d['Label'] = combined_labels
fig = px.scatter_3d(df_3d, x='x', y='y', z='z', color='Label', title="Test + Water + Yogurt (3D Interactive)", opacity=0.7)
fig.show()

### Neural Collapse principle

https://medium.com/@sharadjoshi/what-is-neural-collapse-de1decf83f48

A phenomenon occurring during the final phase of training (Terminal Phase of Training-TPT) where the network continues to optimize beyond zero training error.

This process effectively reduces noise, maximizes intra-cluster separation, and simplifies the final classification decision, ultimately leading to better generalization and adversarial robustness.

Overstepping optimization limits risks overfitting and reduced generalization.

This happens because of the logarithmic loss function, which drives the weight updates during backpropagation to create semantic clusters (one for each class).

The fact that semantically shifted images (Out-of-Distribution, or OOD) form isolated clusters in the latent space is not surprising. It is an inherent behavior of deep neural networks. Ineed, unseen features activate different, and sometimes random, sets of neurons. The consequence is the creation of distinct embeddings projected into different regions of the latent space with respect to the training domain (In-Distribution, or ID)

On the role of neurons in OOD detection (2022)
Density estimation in the latent space (ICML 2020)
Generalized OOD detection via feature space regularization (NeurIPS 2021)

The cluster visualization demonstrates the Neural Collapse phenomenon, the model's feature space exhibits within-class convergence (low variance) and between-class separation (maximized distances), effectively mapping the classes into an optimal geometric structure.

The logarithmic loss function into the CNN produce the embedding collapse into clusters.

The drifted cluster appears isolated and compact within the latent space, suggesting a distinct shift in the underlying data distribution

Execute ML clustering with train embeddings and find out the cluster head of each class distribution rappresenting the semantic anchor into the embeddings vectorial space.

Drift detection is based on euclidean distance to K=1 closest centroid of the train distribution

In [ ]:
data = { #data set of embedding
    "Train": train_embeddings,
    "Val": val_embeddings,
    "Test": test_embeddings,
    "Yogurt": yogurt_embeddings,
    "Water": embeddings_water,
    "Egg": egg_embeddings,
}

In [ ]:
k_range = range(75, 86)
inertias = []
sil_scores = []
for k in k_range:
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42).fit(data["Train"])
    inertias.append(kmeans.inertia_)
    sil_scores.append(silhouette_score(data["Train"], kmeans.labels_))
fig, ax1 = plt.subplots(figsize=(10, 5))
color = 'tab:blue'
ax1.set_xlabel('Number of clusters (K)')
ax1.set_ylabel('Inertia', color=color)
ax1.plot(k_range, inertias, marker='o', color=color, label='Inertia')
ax1.tick_params(axis='y', labelcolor=color)
ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Silhouette Score', color=color)
ax2.plot(k_range, sil_scores, marker='s', color=color, label='Silhouette')
ax2.tick_params(axis='y', labelcolor=color)

plt.title(" Inerzia and Silhouette (K=75 - 85)")
plt.grid(True)
plt.show()

With K Mean there are some difficulties with the best number of clusters

In [ ]:
k_range = range(75, 86)
linkages = ['ward', 'complete', 'average', 'single']
results = []
for linkage in linkages:
    for k in k_range:
        model = AgglomerativeClustering(n_clusters=k, linkage=linkage)
        labels = model.fit_predict(data["Train"])
        score = silhouette_score(data["Train"], labels)
        results.append({"Linkage": linkage, "K": k, "Silhouette": score})
df_results = pd.DataFrame(results)
plt.figure(figsize=(12, 6))
sns.lineplot(data=df_results, x="K", y="Silhouette", hue="Linkage", marker="o")
plt.title("AgglomerativeClustering Silhouette Score between differents Linkage metrics")
plt.grid(True)
plt.show()

AgglomerativeClustering, with ward and average linkage, show up better clustering performance

Avoid DBSCAN because sensible to different clusters density, as in this case with rare classes.

In [ ]:
n_clusters=81
#cluster = KMeans(n_clusters=n_clusters, n_init=10, random_state=42).fit(data["Train"])
#centroids = cluster.cluster_centers_
cluster_model = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
labels = cluster_model.fit_predict(data["Train"])
print(f"{labels[:10]}")

In [ ]:
# Compute cluster centroids by averaging the feature vectors of all data points
# assigned to each specific cluster index (i)
# [labels == i] filter data based on cluster
centroids = np.array([data["Train"][labels == i].mean(axis=0) for i in range(n_clusters)])

In [ ]:
K_MAX = 1  # Change this variable to set the maximum K value of KNN

#train a Kmean only on train set, can be tested also DBscan and AgglomerativeClustering!!!!!!!!!!!!!!!!!!
# DBscan -> i do not know because some algorithms have some issue with cluster that show up differents densiti into data
#kmeans = KMeans(n_clusters=81, n_init=10, random_state=42).fit(data["Train"])
#train_val_combined = np.concatenate([data["Train"], data["Val"]], axis=0)
#kmeans = KMeans(n_clusters=81, n_init=10, random_state=42).fit(train_val_combined)

# exec tsne on all all_data to allow comparison, need the same distribution reduction to all
all_data = np.concatenate([*data.values(), centroids])
all_2d = TSNE(n_components=2, perplexity=50, random_state=42, init="pca").fit_transform(all_data)
###################################################################################
#plot centroid based on trian and val
projections = {}
idx = 0
for name, emb in data.items():
    projections[name] = all_2d[idx:idx + len(emb)]
    idx += len(emb)
centroids_2d = all_2d[idx:]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for i, name in enumerate(["Train", "Val"]):
    axes[i].scatter(projections[name][:, 0], projections[name][:, 1], s=15, alpha=0.4, label=f"{name} Data")
    axes[i].scatter(centroids_2d[:, 0], centroids_2d[:, 1], marker="X", s=100, c="red", edgecolors="black", label="Centroids")
    axes[i].set_title(f"{name} Set & Centroids"); axes[i].legend()
##################################################################################

The validation set has been used for hyperparameter tuning and represents an accurate distribution of the domain variability.

To establish a drift threshold, I compute the interquartile range (IQR) of the distances within this set, the threshold is set to Q3​+1.5×IQR. Filtering out outliers accounting for the natural variance of the data.

Consideration:

- maybe Q3​+1.5×IQR can be a parameter

In [ ]:
knn = NearestNeighbors(n_neighbors=1, metric='euclidean').fit(centroids)
print(knn.metric)
#from sklearn.neighbors import NearestNeighbors
#knn = NearestNeighbors(n_neighbors=1, metric='euclidean')
#knn = NearestNeighbors(n_neighbors=1, metric='cosine')
distances = {}
for name, emb in data.items(): # ex ("train", [3, 4, 5, ...])
    d, _ = knn.kneighbors(emb) #based on centroids -> by default use Minkowski with p=2 as eucleadian distanse =====> runtime complexity O(d*C) number of vector dimension 512 and number of classes 81 but not fixed
    distances[name] = d.flatten()   # 1D array
# Now for each data set collect a distance distribution to K=1 closest centroid
#define a th for drift filtering
val_dists = distances["Val"]
q1, q3 = np.percentile(val_dists, [25, 75])
iqr = q3 - q1
threshold = q3 + 1.5 * iqr
print(f"Threshold computed on Val set: {threshold:.3f}")
############################################################################################
# plot distrances G. like distribution
plt.figure(figsize=(10, 6))
colors = sns.color_palette("viridis", n_colors=len(data))

for i, (label, dist) in enumerate(distances.items()):
    sns.kdeplot(dist, fill=True, alpha=0.2, color=colors[i], label=label)
    # vertical line at the median
    plt.axvline(np.median(dist), color=colors[i], linestyle="--", linewidth=1)

plt.axvline(threshold, color='red', linewidth=2, label=f"Threshold = {threshold:.2f}")
plt.title("Distance to nearest centroid (K=1) – Density plot")
plt.xlabel("Euclidean distance")
plt.ylabel("Density")
plt.legend(fontsize=8)
plt.grid(alpha=0.3)
plt.show()
##########################################################################################
#distences box plots for distribution
rows = []
for name, dist_array in distances.items():
    for d in dist_array:
        rows.append({"Dataset": name, "Distance": d})
df = pd.DataFrame(rows)

plt.figure(figsize=(12, 5))
sns.boxplot(x="Dataset", y="Distance", data=df, palette="viridis", hue="Dataset")
plt.axhline(threshold, color='red', linestyle='--', label=f"Threshold = {threshold:.2f}")
plt.title("Distance distribution (K=1) – Boxplot")
plt.legend()
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.show()

## Analysis

Result are very close to autoencoder approach, but in this case without gpu.

OOD data creates another Gaussian distribution of k-NN (k=1) distances. The separation of the two distributions depends on how the CNN model projects the new and drifted features

--------------------------------------------------------------------------------

This drift detection approach may struggle with samples that mirror the characteristics and distributions of the training domain.

Because of the model's generalization capabilities, it tends to project these out-of-distribution samples into the embedding space near the existing centroids, performing a 'masking' of the drifts.


-------------------------------------------------------------------------------

Egg distribution coverage box (i think) can be close to some training class, maybe to milk box of the model domain.


But the idea is still working well.

## REMEMBER, BEST EFFORT approach

I think it is not possibile execute a perfect drift detection filtering into edge 5G layer without using many computing resources.

Indeed, i prefer having a best effort and robust drift detection policy for data filtering and execute standard dirft detection into cloud.

With this drift detection filter we can reduce the cloud work.


# STAGE 3

Find out some patterns for a dynamic threshold policy

In [ ]:
from sklearn.mixture import GaussianMixture
from scipy.stats import gaussian_kde, norm
# otsu is another algorithm used in CV to detect the separation between two G. di dist, BUT, not so precise when there is a overlap
from skimage.filters import threshold_otsu

In [ ]:
def gmm_threshold(d): #GaussianMixture fit with distances into a buffer and find out the point of division
    gmm=GaussianMixture(n_components=2,random_state=42).fit(d.reshape(-1,1))
    mu,sig,pi=gmm.means_.ravel(),np.sqrt(gmm.covariances_).ravel(),gmm.weights_.ravel()
    o=np.argsort(mu)
    mu1,mu2,s1,s2,p1,p2=mu[o[0]],mu[o[1]],sig[o[0]],sig[o[1]],pi[o[0]],pi[o[1]]
    ## resolution of the equation,
    a=1/(2*s1**2)-1/(2*s2**2)
    b=-mu1/s1**2+mu2/s2**2
    c=mu1**2/(2*s1**2)-mu2**2/(2*s2**2)+np.log((s1*p2)/(s2*p1))
    roots=[-c/b] if abs(a)<1e-12 and abs(b)>1e-12 else []
    if abs(a)>=1e-12:
        disc=b*b-4*a*c
        if disc>=0: roots=[(-b-np.sqrt(disc))/(2*a),(-b+np.sqrt(disc))/(2*a)]

    th=next((r for r in roots if mu1<r<mu2),None)  #distance < th  --> ID distance > th  --> OOD
    #return th,(mu1,s1),(mu2,s2)
    return th

In [ ]:
def distances_to_centroids(X):  #Euclidean distance from each point in X to its nearest centroid.
    return knn.kneighbors(X)[0].flatten()

In [ ]:
np.random.seed(42)   # for reproducible sampling

n_clusters = 81
train_labels = AgglomerativeClustering(n_clusters=n_clusters, linkage="ward").fit_predict(data["Train"])
centroids = np.array([ data["Train"][train_labels == i].mean(axis=0) for i in range(n_clusters) ])

ID = data["Test"]
OOD = np.concatenate([data["Water"], data["Yogurt"], data["Egg"]], axis=0)

print(f"total ID + OOD samples = {len(ID)+len(OOD)}, where ID = {len(ID)} and OOD = {len(OOD)}, OOD: W is {data["Water"].shape[0]}, E is {data["Egg"].shape[0]}, Y is {data["Yogurt"].shape[0]}")
# Distances from validation points to nearest centroid
knn = NearestNeighbors(n_neighbors=1, metric="euclidean").fit(centroids)
val_dist = knn.kneighbors(data["Val"])[0].flatten()

q1, q3 = np.percentile(val_dist, [25, 75])
safe_threshold = q3 + 1.5 * (q3 - q1) #VALIDATION – SAFE THRESHOLD (IQR)
print(f"Safe threshold: {safe_threshold:.3f}")

In [ ]:
#test set up, (buffer dim, ID % into the buffer, OOD% into the buffer)
tests = [  (100, 80, 20),(500, 80, 20), (500, 50, 50), (1000, 90, 10),(2000, 70, 30),
         (300, 95, 5), (400, 10, 90), (1000, 75, 25), (1500, 60, 40), (2500, 85, 15) ]

In [ ]:
fig, axes = plt.subplots(len(tests), 1, figsize=(13, 4.5 * len(tests)))

if len(tests) == 1:
    axes = [axes]

for ax, (buffer_size, id_pct, ood_pct) in zip(axes, tests):

    n_id = min(int(buffer_size * id_pct / 100), len(ID))
    n_ood = min(int(buffer_size * ood_pct / 100), len(OOD))

    X_sample = np.concatenate([ ID[np.random.choice(len(ID), n_id, replace=False)], OOD[np.random.choice(len(OOD), n_ood, replace=False)] ])
    labels = np.concatenate([np.repeat("ID", n_id), np.repeat("OOD", n_ood)])

    shuffled_indices = np.random.permutation(len(X_sample)) # shuffle keeping the correspondences
    X_sample = X_sample[shuffled_indices]
    labels = labels[shuffled_indices]

    dist = distances_to_centroids(X_sample)
    id_dist = dist[labels == "ID"]

    gmm_val = gmm_threshold(dist)
    otsu_val = threshold_otsu(dist)
    id_mean = np.mean(id_dist)
    id_median = np.median(id_dist)
    id_p95 = np.percentile(id_dist, 95)

    x_vals = np.linspace(dist.min(), dist.max(), 400)


    for cls, color in [("ID", "royalblue"), ("OOD", "crimson")]:
        d = dist[labels == cls]
        ax.hist(d, bins=40, density=True, alpha=0.3, color=color, label=f"{cls} hist")
        if len(d) > 1:
            ax.plot(x_vals, gaussian_kde(d)(x_vals), color=color, lw=2, label=f"{cls} KDE")


    lines = [ (gmm_val, "orange", "--", 2, f"GMM={gmm_val:.3f}"), (otsu_val, "purple", ":", 2, f"Otsu={otsu_val:.3f}"), (safe_threshold, "green", "-.", 2, f"Safe={safe_threshold:.3f}"),
        (id_mean, "cyan", "-", 2, f"ID mean={id_mean:.3f}"), (id_median, "blue", "-", 2, f"ID median={id_median:.3f}"), (id_p95, "black", ":", 2, f"ID p95={id_p95:.3f}"),
        (safe_threshold, "red", "-", 4, f"SAFE={safe_threshold:.3f}"), ]

    for val, color, linestyle, linewidth, label in lines:
        ax.axvline(val, color=color, linestyle=linestyle, linewidth=linewidth, label=label)

    ax.axvspan(safe_threshold, dist.max(), color="red", alpha=0.08)

    ax.set_title(f"Buffer={len(X_sample)} | ID={id_pct}% OOD={ood_pct}%")
    ax.set_xlabel("Distance to nearest train centroid")
    ax.set_ylabel("Density")
    ax.legend(fontsize=8, loc="upper right")
    ax.grid(alpha=0.35)

plt.tight_layout()
plt.show()

While a SAFE threshold based on the validation set distribution is a valid approach for a static filtering policy, a more dynamic solution might be required for better adaptability

When the percentage of Out-of-Distribution (OOD) samples is low, it becomes difficult to filter OOD data with high precision without misclassify In-Distribution (ID) samples

Buffer dimensioning is critical because with a small sample size, it is statistically challenging to accurately distinguish between the two distributions

GaussianMixture.fit() is not ideal for a hard-real-time operation. However, implementing a sample batch policy could optimize its performance for streaming or real-time environments

# STAGE 4

I am looking for a dynamic GMM thresholding method that adjusts threshold around a 'SAFE' value

## RealTimeOODDetector class

Perform soft real-time Out-of-Distribution (OOD) detection. Use an adaptive threshold to determine whether incoming data is  closer to a known In-Distribution (ID) data or new OOD data.

Dynamically computes the threshold based on historical distance distributions.

The threshold is derived into two distinct Gaussian distance distributions:

  - One representing In-Distribution (ID) data.
  - One representing Out-of-Distribution (OOD) data.

The proposed system can be considered a soft real-time solution. It does not guarantee hard real-time constraints because the periodic gmm.fit() operation introduces latency spikes that may exceed the hard real time deadline.

 Increasing the batch size reduces the number of threshold updates reducing the threshold relaibility, but also improves the overall real-time performance and bringing the system closer to hard real-time behavior

In [ ]:
class RealTimeOODDetector:
    def __init__(self, centroids, window_size=400, batch_size=20, initial_threshold=None, smoothing=0.9, min_samples_gmm=50, min_mean_gap_std=1.0, max_percentile=99, safe_threshold=None, max_consecutive_drops=5):
        self.knn = NearestNeighbors(n_neighbors=1, metric="euclidean")
        self.knn.fit(centroids) # latent space semantic anchors
        # distance distribution history composed by a FIFO queue
        self.distance_buffer = deque(maxlen=window_size) # A big window allow to keep a better estimated view of the distribution of distances
        self.batch_distances = []

        self.threshold = initial_threshold # threshold = q3 + 1.5 * iqr (of validation set). it is the th starting point
        self.safe_threshold = safe_threshold if safe_threshold is not None else initial_threshold

        self.batch_size = batch_size # frequency of threshold update, trade of between cpu overhead and reliability of estimates over time
        self.smoothing = smoothing
        self.min_samples_gmm = min_samples_gmm
        self.min_mean_gap_std = min_mean_gap_std
        self.max_percentile = max_percentile

        self.max_consecutive_drops = max_consecutive_drops
        self.consecutive_drops = 0
        # for the stable fluctuation around a safety value of the threshold
        self.threshold_history = []
        self.threshold_trend = []
        self.counter = 0
        self.th_issue=0

        self.history = {"time":[],"distances":[],"thresholds":[],"predictions":[],"ground_truth":[],"trend":[], "th_issue":[]}

    def gaussian_boundary_threshold(self, distances): #distances variabile is the distances history, this method use G. M. to detect the division point as new threshold value over time
        if len(distances) < self.min_samples_gmm: # is the buffer is too small avoid
            return None
        try:
            gmm = GaussianMixture(n_components=2, covariance_type="full", n_init=5, random_state=42) # assume to have two G. distance distributions
            gmm.fit(distances.reshape(-1,1)) # heavy operation, soft real time, can be mitigate by batching policy

            mu = gmm.means_.flatten()
            sigma = np.sqrt(gmm.covariances_.flatten())
            pi = gmm.weights_.flatten()

            idx = np.argsort(mu)
            mu1, mu2 = mu[idx]
            s1, s2 = sigma[idx]
            p1, p2 = pi[idx]

            if np.mean([s1,s2]) <= 0 or (mu2-mu1)/np.mean([s1,s2]) < self.min_mean_gap_std: #is the two G. are detected too close avoid
                return None

            a = 1/(2*s1**2)-1/(2*s2**2)
            b = -mu1/(s1**2)+mu2/(s2**2)
            c = mu1**2/(2*s1**2)-mu2**2/(2*s2**2)+np.log((s1*p2)/(s2*p1))

            roots = [-c/b] if abs(a)<1e-12 and abs(b)>1e-12 else []

            if abs(a)>=1e-12:
                disc = b*b-4*a*c
                if disc >= 0:
                    roots = [(-b-np.sqrt(disc))/(2*a),(-b+np.sqrt(disc))/(2*a)]

            valid = [r for r in roots if mu1 < r < mu2]
            return float(valid[0]) if valid else None  # return the th, point of the separation between ID and OOD G. distribution, the discriminator for classification

        except:
            return None

    def _update_threshold(self): #methods to update threshold value over time, there are some filter for QoS
        if len(self.distance_buffer) < self.min_samples_gmm: #check the numer of sample
            return
        buffer = np.asarray(self.distance_buffer)
        thr = self.gaussian_boundary_threshold(buffer)

        if thr is None: #if there are some th isseu use a fallback policy
            self.th_issue+=1
            thr = np.mean(buffer)+2*np.std(buffer) # the fallback threshold is -> mean+2*std of the distance history

        thr = min(thr,np.percentile(buffer,self.max_percentile)) # avoid outliers threshold values
        new_thr = thr if self.threshold is None else self.smoothing*self.threshold+(1-self.smoothing)*thr #smoothing of threshold update, avoid spike

        if self.threshold is not None: # avoid threshold collapse in case of a pure ID distribution

            if new_thr < self.threshold:
                self.consecutive_drops += 1
                trend = "DOWN"
            elif new_thr > self.threshold:
                self.consecutive_drops = 0
                trend = "UP"
            else:
                trend = "SAME"
            # after too long sequence of threshold decrease reset it to avoid the collapse
            if self.consecutive_drops >= self.max_consecutive_drops and self.safe_threshold is not None:
                new_thr = self.safe_threshold
                self.consecutive_drops = 0
                trend = "RESET"
        else:
            trend = "INIT"

        self.threshold = new_thr
        self.threshold_history.append(new_thr)
        self.threshold_trend.append(trend)

    def process(self, embedding, true_label=None):

        dist = self.knn.kneighbors(np.asarray(embedding).reshape(1,-1))[0][0][0] #find out the closest semantic anchor center and get the distance

        self.distance_buffer.append(dist)
        self.batch_distances.append(dist)
        self.counter += 1

        if len(self.batch_distances) >= self.batch_size:
            self._update_threshold()
            self.batch_distances.clear()

        if self.threshold is None and len(self.distance_buffer)>=self.min_samples_gmm:
            print("\n exceed ")
            self.threshold = np.percentile(self.distance_buffer,self.max_percentile)

        is_ood = self.threshold is not None and dist > self.threshold #classification operation

        trend = self.threshold_trend[-1] if self.threshold_trend else "INIT" #if threshold_trend is none assign INIT, otherwise load the last values detected
        #loggiing of the class service
        self.history["time"].append(self.counter-1)
        self.history["distances"].append(dist)
        self.history["thresholds"].append(self.threshold if self.threshold is not None else np.nan)
        self.history["predictions"].append(int(is_ood)) # system forecast
        self.history["ground_truth"].append(0 if true_label is None else true_label) # real data classification
        self.history["trend"].append(trend)
        #self.history["th_isseu"].append(self.th_issue)

        return dist,is_ood,self.threshold

## System performance stress-test

A testing phase to evaluate the overall performance of the system using two distinct stream-based workflows:

* **Randomly Sampled Streams:** Data streams composed of a random mix of ID and OOD data, evaluated across varying percentage ratios.
* **Wave-Based Streams:** Data streams structured in sequential "waves" of ID and/or OOD data, also tested using different percentage distributions.

In [ ]:
def generate_random_stream(data, n_test, n_ood, replace=False): #ID and/or OOD are stream noise into the incoming signal
    test_embs = data["Test"] # ID → data["Test"]
    ood_embs = np.concatenate( [data["Water"], data["Yogurt"], data["Egg"]] ) # OOD → Water, Yogurt, Egg
    # security
    n_test = min(n_test, len(test_embs))
    n_ood = min(n_ood, len(ood_embs))
    # random sampling with or with out replacement
    sampled_test = test_embs[np.random.choice(len(test_embs), n_test, replace=replace)]
    sampled_ood = ood_embs[np.random.choice(len(ood_embs), n_ood, replace=replace)]

    combined = np.concatenate([sampled_test, sampled_ood])
    labels = np.concatenate([np.zeros(n_test, dtype=int), np.ones(n_ood, dtype=int)]) #labels 0 = ID 1 = OOD

    perm = np.random.permutation(len(combined)) # random shuffle, creare a stream signal with noise inside

    return combined[perm], labels[perm]
# wave_pattern-> 'Standard': ['Test', 'Yogurt', 'Water', 'Egg'], 'Binary': ['Test', 'Yogurt', 'Test', 'Egg']
# block_size= [20, 30, 50, 100 ....]
def build_wave_stream(data, wave_pattern, block_size, stream_length): #creare waves of same data class

    all_data, all_labels = [], []

    indices = {name: 0 for name in set(wave_pattern) }  #start a counter for each wave_pattern

    while len(all_data) < stream_length:

        for name in wave_pattern: # this operation restart until reach the stream_length limit

            emb = data[name]
            n_emb = len(emb)

            for _ in range(block_size): #extract the block size of the class

                sample = emb[ indices[name] % n_emb ] #avoid possible out of range
                all_data.append(sample)
                all_labels.append(name)
                indices[name] += 1 #update the sample value of the class

                if len(all_data) >= stream_length:
                    break

            if len(all_data) >= stream_length:
                break

    return np.array(all_data), np.array(all_labels)

Plot the confusion matrix and distance metrix evolution with the threshold for each test

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title, ax):
    ConfusionMatrixDisplay.from_predictions(y_true, y_pred, ax=ax, cmap='Blues')
    ax.set_title(title)

def plot_temporal(time, dist, thresh, pred, true, title, ax1, ax2, ax3):
    ax1.plot(time, dist, 'b.', alpha=0.3, label='Distance')
    ax1.plot(time, thresh, 'r-', lw=2, label='Threshold')
    anom_idx = np.where(pred == 1)[0]
    if len(anom_idx) > 0:
        ax1.scatter(anom_idx, dist[anom_idx], color='red', s=50, marker='^', label='OOD detected')
    true_idx = np.where(true == 1)[0]
    if len(true_idx) > 0:
        ax1.scatter(true_idx, dist[true_idx], color='green', s=30, marker='o', facecolors='none', label='Ground truth OOD')
    ax1.legend()
    ax1.set_title(title)
    ax1.set_ylabel('Distance')
    ax1.grid(alpha=0.3)

    ax2.plot(time, thresh, 'g-', lw=2)
    ax2.set_ylabel('Threshold')
    ax2.grid(alpha=0.3)

    ax3.plot(time, dist, 'b-', alpha=0.6, label='Distance')
    ax3.plot(time, thresh, 'r--', label='Threshold')
    ax3.set_xlabel('Time (samples)')
    ax3.set_ylabel('Distance')
    ax3.legend()
    ax3.grid(alpha=0.3)

In [ ]:
def start_test_phase(stream_data, stream_labels, detector_params, nome_test, val_dists):

    if isinstance(stream_labels[0], (str, np.str_)): #labels chack
        print(f"\n string labels input, convert to int data type")
        stream_labels = np.array([0 if lab == 'Test' else 1 for lab in stream_labels])
    else:
        print(f"\n all int data types")
        stream_labels = np.array(stream_labels, dtype=int)


    detector =RealTimeOODDetector(**detector_params)
    # Warm-up fill buffer with validation set sample
    for d in val_dists[:300]:
        detector.distance_buffer.append(d)

    for emb, lab in zip(stream_data, stream_labels): # creation of the embedding data stream
        detector.process(emb, true_label=int(lab)) #detector receives embedding and true label 0-1 of the sample

    if len(detector.batch_distances) > 0:
       detector._update_threshold()
    #plot the result of the stream that get in to the detector
    hist = detector.history
    time_steps = np.array(hist['time'])
    distances = np.array(hist['distances'])
    thresholds = np.array(hist['thresholds'])

    predictions = np.array(hist['predictions']) ####
    ground_truth = np.array(hist['ground_truth']) ####

    valid = ~np.isnan(thresholds) # cleaning mask
    time_valid = time_steps[valid]
    dist_valid = distances[valid]
    th_valid = thresholds[valid]

    pred_valid = predictions[valid]
    true_valid = ground_truth[valid].astype(int)

    if np.sum(pred_valid) == 0:
        print("\n ERROR: there was no OOD!")
        acc = accuracy_score(true_valid, pred_valid)
        prec = rec = f1 = 0.0
    else:
        acc = accuracy_score(true_valid, pred_valid) #sample with a correctly prediction
        #pos_label=1 ----> OOD class
        prec = precision_score(true_valid, pred_valid, pos_label=1, zero_division=0) #how often the detector that predict OOD have right
        rec = recall_score(true_valid, pred_valid, pos_label=1, zero_division=0) #how many OOD the system can detect
        f1 = f1_score(true_valid, pred_valid, pos_label=1, zero_division=0) #armonic mean between accuracy and precision

    print("\n" + "="*60)
    print(f"RISULTATI TEST: {nome_test}")
    print("="*60)
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision (OOD): {prec:.4f}")
    print(f"Recall (OOD):    {rec:.4f}")
    print(f"F1-score (OOD):  {f1:.4f}")
    print("\n classification report:")
    print(classification_report(true_valid, pred_valid, labels=[0, 1], target_names=['ID', 'OOD'], zero_division=0))

    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
    plot_temporal(time_valid, dist_valid, th_valid, pred_valid, true_valid, f"{nome_test} - Acc={acc:.3f}, F1={f1:.3f}", ax1, ax2, ax3)
    plt.tight_layout()
    plt.show()

    fig2, ax = plt.subplots(figsize=(6, 5))
    plot_confusion_matrix(true_valid, pred_valid, f" confusion matrix - {nome_test}", ax)
    plt.tight_layout()
    plt.show()

    return {'acc': acc, 'prec': prec, 'rec': rec, 'f1': f1}

In [ ]:
REPO_ID = "tomasconti/Drift_Detection"
print("Downloading native 512-D embeddings from HuggingFace")

path_train = hf_hub_download(repo_id=REPO_ID, filename="good_to_delivery/grocery_train_embeddings.npy")
path_yogurt = hf_hub_download(repo_id=REPO_ID, filename="good_to_delivery/yogurt_embeddings.npy")
path_test = hf_hub_download(repo_id=REPO_ID, filename="good_to_delivery/grocery_test_embeddings.npy")
path_val = hf_hub_download(repo_id=REPO_ID, filename="good_to_delivery/grocery_val_embeddings.npy")
path_water = hf_hub_download(repo_id=REPO_ID, filename="good_to_delivery/still_water_embeddings.npy")
path_egg = hf_hub_download(repo_id=REPO_ID, filename="good_to_delivery/egg_embeddings.npy")

test_embeddings = np.load(path_test)
val_embeddings = np.load(path_val)
train_embeddings = np.load(path_train)
embeddings_water = np.load(path_water)
yogurt_embeddings = np.load(path_yogurt)
egg_embeddings = np.load(path_egg)

print("Embeddings shapes:")
print(f"Train: {train_embeddings.shape}")
print(f"Val:   {val_embeddings.shape}")
print(f"Test:  {test_embeddings.shape}")
print(f"Water: {embeddings_water.shape}")
print(f"Yogurt:{yogurt_embeddings.shape}")
print(f"Egg:   {egg_embeddings.shape}")

data = { "Train": train_embeddings, "Val": val_embeddings, "Test": test_embeddings, "Yogurt": yogurt_embeddings, "Water": embeddings_water, "Egg": egg_embeddings }

n_clusters = 81
cluster_model = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
labels = cluster_model.fit_predict(data["Train"])
centroids = np.array([data["Train"][labels == i].mean(axis=0) for i in range(n_clusters)])

knn_val = NearestNeighbors(n_neighbors=1, metric='euclidean').fit(centroids)
val_dists, _ = knn_val.kneighbors(data["Val"])
val_dists = val_dists.flatten()
q1, q3 = np.percentile(val_dists, [25, 75])
iqr = q3 - q1
initial_threshold = q3 + 1.5 * iqr # base the starting and safe threshold on validation set because representative of the domain but only used for hyperparameter tuning and not for weights
print(f"\nIQR starting threshold : {initial_threshold:.3f}\n")
#TOTAL_SAMPLES=2400
TOTAL_SAMPLES = egg_embeddings.shape[0]+yogurt_embeddings.shape[0]+embeddings_water.shape[0]+test_embeddings.shape[0]

"""
detector_params = { 'centroids': centroids, 'window_size': 400,'batch_size': 50,'initial_threshold': initial_threshold }
"""
detector_params = { # detector parameters
    "centroids": centroids,
    "window_size": 400,
    "batch_size": 20,
    "initial_threshold": initial_threshold,
    "smoothing": 0.9,
    "min_samples_gmm": 50,
    "min_mean_gap_std": 1.0,
    "max_percentile": 99,
    "safe_threshold": initial_threshold,
    "max_consecutive_drops": 5
}
# ID/OOD % ratios
ratio_percentages = [ (0.95, 0.05), (0.90, 0.10), (0.80, 0.20), (0.70, 0.30), (0.60, 0.40), (0.50, 0.50), ]

block_sizes = [50, 100, 200] # each blocks is a waves of data
wave_patterns = { 'Standard': ['Test', 'Yogurt', 'Water', 'Egg'], 'FastSwitch': ['Test', 'Water', 'Yogurt', 'Egg', 'Test', 'Yogurt'] } #waves stream patters

tests = []

# Random streams
for id_pct, ood_pct in ratio_percentages:
    n_id = int(TOTAL_SAMPLES * id_pct)
    n_ood = int(TOTAL_SAMPLES * ood_pct)
    for replace in [True, False]: # sampling form the ebedding pool can be done with or with out replacement
        label = f"Random {'w/' if replace else 'w/o'} Rep ({id_pct*100:.0f}%ID/{ood_pct*100:.0f}%OOD)"
        #creation of all the possibile parameter combination of the random stream generator
        tests.append( ( label, lambda ni=n_id, no=n_ood, r=replace: generate_random_stream(data, n_test=ni, n_ood=no, replace=r), detector_params ) )

# Wave streams
for pattern_name, pattern in wave_patterns.items():
    for b_size in block_sizes: # for each waves stream patter try all block combination
        label = f"Wave {pattern_name} Block{b_size}"
        #creation of all the possibile parameter combination of the build_wave_stream
        tests.append( (label, lambda b=b_size, pat=pattern: build_wave_stream(data, wave_pattern=pat, block_size=b, stream_length=TOTAL_SAMPLES), detector_params ))

print(f"Numer of tests: {len(tests)}")

results = {}
for name, generator, params in tests:
    stream_data, stream_labels = generator()
    res = start_test_phase(stream_data, stream_labels, params, name, val_dists)
    results[name] = res

print("\n" + "="*60)
print(" RECAP ")
print("="*60)
print(f"{'Test':<45} {'Acc':>8} {'Prec':>8} {'Rec':>8} {'F1':>8}")
for name, res in results.items():
    print(f"{name:<45} {res['acc']:8.4f} {res['prec']:8.4f} {res['rec']:8.4f} {res['f1']:8.4f}")

random_tests = [k for k in results if "Random" in k]
wave_tests = [k for k in results if "Wave" in k]

def stats_gruppo(names, label):
    if not names:
        return
    acc  = [results[n]['acc'] for n in names]
    prec = [results[n]['prec'] for n in names]
    rec  = [results[n]['rec'] for n in names]
    f1   = [results[n]['f1'] for n in names]
    print(f"\n--- {label} ---")
    print(f"Acc: {np.mean(acc):.4f} ± {np.std(acc):.4f}")
    print(f"Prec: {np.mean(prec):.4f} ± {np.std(prec):.4f}")
    print(f"Rec:  {np.mean(rec):.4f} ± {np.std(rec):.4f}")
    print(f"F1:   {np.mean(f1):.4f} ± {np.std(f1):.4f}")

stats_gruppo(random_tests, "Test Random (basic)")
stats_gruppo(wave_tests, "Test Wave (basic)")

acc_list = [res['acc'] for res in results.values()]
prec_list = [res['prec'] for res in results.values()]
rec_list = [res['rec'] for res in results.values()]
f1_list = [res['f1'] for res in results.values()]

acc_mean, acc_std = np.mean(acc_list), np.std(acc_list)
prec_mean, prec_std = np.mean(prec_list), np.std(prec_list)
rec_mean, rec_std = np.mean(rec_list), np.std(rec_list)
f1_mean, f1_std = np.mean(f1_list), np.std(f1_list)

print("\n" + "="*60)
print(f"Average accuracy over all tests: {acc_mean:.4f} ± {acc_std:.4f}")
print(f"Average (OOD) precision over all tests : {prec_mean:.4f} ± {prec_std:.4f}")
print(f"Average (OOD) recall over all tests : {rec_mean:.4f} ± {rec_std:.4f}")
print(f"Average (OOD) F1-score over all the tests: {f1_mean:.4f} ± {f1_std:.4f}")

metriche = ['Accuracy', 'Precision', 'Recall', 'F1']
medie = [acc_mean, prec_mean, rec_mean, f1_mean]
std_dev = [acc_std, prec_std, rec_std, f1_std]

fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.bar(metriche, medie, yerr=std_dev, capsize=5, color=['skyblue', 'lightgreen', 'lightcoral', 'gold'], alpha=0.8, edgecolor='black')
ax.set_ylabel('Average value')
ax.set_title(' Average performances \n(Error bars represent standard deviation) ')
ax.set_ylim([0, 1])
ax.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, medie):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{val:.3f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()



# Massive test

Result coming from a massive test with different filter setting and random-waves streams configurations

The large-scale evaluation demonstrates that the proposed adaptive OOD detector achieves stable performance across different stream types and parameter configurations.

For **random streams**, the detector obtains an average accuracy of **0.911 ± 0.018** and an OOD F1-score of **0.820 ± 0.147**. Although the recall remains very high (0.952 ± 0.011), the lower precision indicates a higher number of false positives. This behavior is expected, as the detector prefer detect OOD sample even if with some ID false positive classificaiton (trade off, better accuracy but less precision).

For **wave streams**, performance improves significantly, reaching an average accuracy of **0.919 ± 0.020** and an **OOD F1-score of 0.933 ± 0.023**. This suggests that the adaptive threshold is more effective when the data distribution changes gradually.

Across all tested window sizes and batch sizes, the performance remains remarkably consistent (average accuracy ≈ **0.914**, average OOD F1-score ≈ **0.866**), indicating that the method is robust to parameter selection while maintaining a high OOD detection rate.

On average a 49.99% of the stream has been filtered.

 ### Allocate storage and computing power for model retrain on image classes already classified well is a waste of resources, it is better to optimize instead.

In [ ]:
""
IQR starting threshold : 13.319
Total number of tests: 333
Test                                                    Acc     Prec      Rec       F1
Random ws=200 bs=20 w/ Rep (95%ID/5%OOD)             0.8800   0.2921   0.9833   0.4504
Random ws=200 bs=20 w/o Rep (95%ID/5%OOD)            0.8746   0.2754   0.9250   0.4245
Random ws=200 bs=20 w/ Rep (90%ID/10%OOD)            0.8883   0.4711   0.9500   0.6298
Random ws=200 bs=20 w/o Rep (90%ID/10%OOD)           0.8867   0.4668   0.9375   0.6233
Random ws=200 bs=20 w/ Rep (80%ID/20%OOD)            0.9033   0.6862   0.9521   0.7976
Random ws=200 bs=20 w/o Rep (80%ID/20%OOD)           0.9083   0.6982   0.9542   0.8063
Random ws=200 bs=20 w/ Rep (70%ID/30%OOD)            0.8980   0.7106   0.9336   0.8070
Random ws=200 bs=20 w/o Rep (70%ID/30%OOD)           0.9113   0.7360   0.9537   0.8309
Random ws=200 bs=20 w/ Rep (60%ID/40%OOD)            0.9153   0.7699   0.9557   0.8528
Random ws=200 bs=20 w/o Rep (60%ID/40%OOD)           0.9102   0.7618   0.9457   0.8438
Random ws=200 bs=20 w/ Rep (50%ID/50%OOD)            0.9081   0.7885   0.9376   0.8566
Random ws=200 bs=20 w/o Rep (50%ID/50%OOD)           0.9146   0.7963   0.9517   0.8671
Random ws=200 bs=20 w/ Rep (40%ID/60%OOD)            0.9080   0.8212   0.9336   0.8738
Random ws=200 bs=20 w/o Rep (40%ID/60%OOD)           0.9135   0.8237   0.9497   0.8822
Random ws=200 bs=20 w/ Rep (30%ID/70%OOD)            0.9170   0.8626   0.9477   0.9032
Random ws=200 bs=20 w/o Rep (30%ID/70%OOD)           0.9293   0.8870   0.9477   0.9163
Random ws=200 bs=20 w/ Rep (20%ID/80%OOD)            0.8966   0.8779   0.9256   0.9011
Random ws=200 bs=20 w/o Rep (20%ID/80%OOD)           0.9232   0.9058   0.9477   0.9263
Random ws=200 bs=20 w/ Rep (10%ID/90%OOD)            0.9335   0.9647   0.9356   0.9499
Random ws=200 bs=20 w/o Rep (10%ID/90%OOD)           0.9267   0.9530   0.9376   0.9452
Random ws=200 bs=20 w/ Rep (5%ID/95%OOD)             0.9433   0.9695   0.9598   0.9646
Random ws=200 bs=20 w/o Rep (5%ID/95%OOD)            0.9465   0.9677   0.9658   0.9668
Random ws=200 bs=50 w/ Rep (95%ID/5%OOD)             0.8862   0.3003   0.9583   0.4573
Random ws=200 bs=50 w/o Rep (95%ID/5%OOD)            0.8867   0.3031   0.9750   0.4625
Random ws=200 bs=50 w/ Rep (90%ID/10%OOD)            0.8942   0.4854   0.9667   0.6462
Random ws=200 bs=50 w/o Rep (90%ID/10%OOD)           0.8858   0.4659   0.9667   0.6287
Random ws=200 bs=50 w/ Rep (80%ID/20%OOD)            0.8979   0.6705   0.9625   0.7904
Random ws=200 bs=50 w/o Rep (80%ID/20%OOD)           0.9067   0.6939   0.9542   0.8035
Random ws=200 bs=50 w/ Rep (70%ID/30%OOD)            0.8934   0.6934   0.9557   0.8037
Random ws=200 bs=50 w/o Rep (70%ID/30%OOD)           0.9049   0.7245   0.9416   0.8189
Random ws=200 bs=50 w/ Rep (60%ID/40%OOD)            0.8998   0.7432   0.9316   0.8268
Random ws=200 bs=50 w/o Rep (60%ID/40%OOD)           0.9066   0.7500   0.9537   0.8397
Random ws=200 bs=50 w/ Rep (50%ID/50%OOD)            0.9146   0.7963   0.9517   0.8671
Random ws=200 bs=50 w/o Rep (50%ID/50%OOD)           0.9093   0.7854   0.9497   0.8597
Random ws=200 bs=50 w/ Rep (40%ID/60%OOD)            0.8950   0.7783   0.9678   0.8628
Random ws=200 bs=50 w/o Rep (40%ID/60%OOD)           0.9252   0.8527   0.9437   0.8959
Random ws=200 bs=50 w/ Rep (30%ID/70%OOD)            0.9277   0.8725   0.9638   0.9159
Random ws=200 bs=50 w/o Rep (30%ID/70%OOD)           0.9211   0.8720   0.9457   0.9073
Random ws=200 bs=50 w/ Rep (20%ID/80%OOD)            0.9099   0.8794   0.9537   0.9151
Random ws=200 bs=50 w/o Rep (20%ID/80%OOD)           0.9171   0.9000   0.9416   0.9204
Random ws=200 bs=50 w/ Rep (10%ID/90%OOD)            0.9322   0.9497   0.9497   0.9497
Random ws=200 bs=50 w/o Rep (10%ID/90%OOD)           0.9349   0.9428   0.9618   0.9522
Random ws=200 bs=50 w/ Rep (5%ID/95%OOD)             0.9433   0.9695   0.9598   0.9646
Random ws=200 bs=50 w/o Rep (5%ID/95%OOD)            0.9449   0.9833   0.9477   0.9652
Random ws=200 bs=100 w/ Rep (95%ID/5%OOD)            0.8783   0.2839   0.9417   0.4363
Random ws=200 bs=100 w/o Rep (95%ID/5%OOD)           0.8829   0.2931   0.9500   0.4479
Random ws=200 bs=100 w/ Rep (90%ID/10%OOD)           0.8888   0.4716   0.9333   0.6266
Random ws=200 bs=100 w/o Rep (90%ID/10%OOD)          0.8933   0.4831   0.9542   0.6415
Random ws=200 bs=100 w/ Rep (80%ID/20%OOD)           0.8896   0.6503   0.9688   0.7782
Random ws=200 bs=100 w/o Rep (80%ID/20%OOD)          0.8946   0.6643   0.9563   0.7839
Random ws=200 bs=100 w/ Rep (70%ID/30%OOD)           0.8994   0.7056   0.9598   0.8133
Random ws=200 bs=100 w/o Rep (70%ID/30%OOD)          0.9022   0.7113   0.9618   0.8178
Random ws=200 bs=100 w/ Rep (60%ID/40%OOD)           0.8911   0.7109   0.9698   0.8204
Random ws=200 bs=100 w/o Rep (60%ID/40%OOD)          0.9009   0.7350   0.9598   0.8325
Random ws=200 bs=100 w/ Rep (50%ID/50%OOD)           0.9140   0.7940   0.9537   0.8665
Random ws=200 bs=100 w/o Rep (50%ID/50%OOD)          0.9157   0.7960   0.9577   0.8694
Random ws=200 bs=100 w/ Rep (40%ID/60%OOD)           0.9142   0.8207   0.9577   0.8839
Random ws=200 bs=100 w/o Rep (40%ID/60%OOD)          0.9163   0.8295   0.9497   0.8856
Random ws=200 bs=100 w/ Rep (30%ID/70%OOD)           0.9170   0.8523   0.9638   0.9046
Random ws=200 bs=100 w/o Rep (30%ID/70%OOD)          0.9269   0.8835   0.9457   0.9135
Random ws=200 bs=100 w/ Rep (20%ID/80%OOD)           0.9089   0.8835   0.9457   0.9135
Random ws=200 bs=100 w/o Rep (20%ID/80%OOD)          0.9304   0.9214   0.9437   0.9324
Random ws=200 bs=100 w/ Rep (10%ID/90%OOD)           0.9294   0.9477   0.9477   0.9477
Random ws=200 bs=100 w/o Rep (10%ID/90%OOD)          0.9308   0.9514   0.9457   0.9485
Random ws=200 bs=100 w/ Rep (5%ID/95%OOD)            0.9352   0.9790   0.9396   0.9589
Random ws=200 bs=100 w/o Rep (5%ID/95%OOD)           0.9352   0.9790   0.9396   0.9589
Random ws=400 bs=20 w/ Rep (95%ID/5%OOD)             0.8762   0.2857   0.9833   0.4428
Random ws=400 bs=20 w/o Rep (95%ID/5%OOD)            0.8842   0.2964   0.9583   0.4528
Random ws=400 bs=20 w/ Rep (90%ID/10%OOD)            0.8938   0.4843   0.9625   0.6444
Random ws=400 bs=20 w/o Rep (90%ID/10%OOD)           0.8904   0.4764   0.9667   0.6382
Random ws=400 bs=20 w/ Rep (80%ID/20%OOD)            0.9058   0.6896   0.9625   0.8035
Random ws=400 bs=20 w/o Rep (80%ID/20%OOD)           0.9038   0.6867   0.9542   0.7986
Random ws=400 bs=20 w/ Rep (70%ID/30%OOD)            0.9100   0.7298   0.9618   0.8299
Random ws=400 bs=20 w/o Rep (70%ID/30%OOD)           0.9063   0.7216   0.9598   0.8238
Random ws=400 bs=20 w/ Rep (60%ID/40%OOD)            0.9164   0.7741   0.9517   0.8538
Random ws=400 bs=20 w/o Rep (60%ID/40%OOD)           0.9060   0.7480   0.9557   0.8392
Random ws=400 bs=20 w/ Rep (50%ID/50%OOD)            0.9210   0.8190   0.9376   0.8743
Random ws=400 bs=20 w/o Rep (50%ID/50%OOD)           0.9051   0.7745   0.9537   0.8548
Random ws=400 bs=20 w/ Rep (40%ID/60%OOD)            0.9121   0.8336   0.9276   0.8781
Random ws=400 bs=20 w/o Rep (40%ID/60%OOD)           0.9128   0.8280   0.9396   0.8803
Random ws=400 bs=20 w/ Rep (30%ID/70%OOD)            0.9277   0.8752   0.9598   0.9155
Random ws=400 bs=20 w/o Rep (30%ID/70%OOD)           0.9154   0.8595   0.9477   0.9014
Random ws=400 bs=20 w/ Rep (20%ID/80%OOD)            0.9181   0.9033   0.9396   0.9211
Random ws=400 bs=20 w/o Rep (20%ID/80%OOD)           0.9130   0.8887   0.9477   0.9172
Random ws=400 bs=20 w/ Rep (10%ID/90%OOD)            0.9349   0.9481   0.9557   0.9519
Random ws=400 bs=20 w/o Rep (10%ID/90%OOD)           0.9240   0.9509   0.9356   0.9432
Random ws=400 bs=20 w/ Rep (5%ID/95%OOD)             0.9254   0.9727   0.9336   0.9528
Random ws=400 bs=20 w/o Rep (5%ID/95%OOD)            0.9433   0.9833   0.9457   0.9641
Random ws=400 bs=50 w/ Rep (95%ID/5%OOD)             0.8804   0.2907   0.9667   0.4470
Random ws=400 bs=50 w/o Rep (95%ID/5%OOD)            0.8817   0.2919   0.9583   0.4475
Random ws=400 bs=50 w/ Rep (90%ID/10%OOD)            0.9004   0.5011   0.9708   0.6610
Random ws=400 bs=50 w/o Rep (90%ID/10%OOD)           0.8908   0.4773   0.9625   0.6381
Random ws=400 bs=50 w/ Rep (80%ID/20%OOD)            0.9096   0.7008   0.9563   0.8088
Random ws=400 bs=50 w/o Rep (80%ID/20%OOD)           0.9038   0.6855   0.9583   0.7993
Random ws=400 bs=50 w/ Rep (70%ID/30%OOD)            0.8879   0.6841   0.9457   0.7939
Random ws=400 bs=50 w/o Rep (70%ID/30%OOD)           0.9045   0.7227   0.9437   0.8185
Random ws=400 bs=50 w/ Rep (60%ID/40%OOD)            0.9066   0.7524   0.9477   0.8388
Random ws=400 bs=50 w/o Rep (60%ID/40%OOD)           0.9117   0.7604   0.9577   0.8477
Random ws=400 bs=50 w/ Rep (50%ID/50%OOD)            0.9193   0.8010   0.9638   0.8749
Random ws=400 bs=50 w/o Rep (50%ID/50%OOD)           0.9093   0.7807   0.9598   0.8610
Random ws=400 bs=50 w/ Rep (40%ID/60%OOD)            0.9128   0.8426   0.9155   0.8775
Random ws=400 bs=50 w/o Rep (40%ID/60%OOD)           0.9128   0.8223   0.9497   0.8814
Random ws=400 bs=50 w/ Rep (30%ID/70%OOD)            0.9310   0.8831   0.9577   0.9189
Random ws=400 bs=50 w/o Rep (30%ID/70%OOD)           0.9137   0.8525   0.9537   0.9003
Random ws=400 bs=50 w/ Rep (20%ID/80%OOD)            0.9417   0.9183   0.9718   0.9443
Random ws=400 bs=50 w/o Rep (20%ID/80%OOD)           0.9150   0.8848   0.9577   0.9198
Random ws=400 bs=50 w/ Rep (10%ID/90%OOD)            0.9335   0.9480   0.9537   0.9509
Random ws=400 bs=50 w/o Rep (10%ID/90%OOD)           0.9267   0.9421   0.9497   0.9459
Random ws=400 bs=50 w/ Rep (5%ID/95%OOD)             0.9465   0.9754   0.9577   0.9665
Random ws=400 bs=50 w/o Rep (5%ID/95%OOD)            0.9498   0.9717   0.9658   0.9687
Random ws=400 bs=100 w/ Rep (95%ID/5%OOD)            0.8742   0.2780   0.9500   0.4302
Random ws=400 bs=100 w/o Rep (95%ID/5%OOD)           0.8825   0.2955   0.9750   0.4535
Random ws=400 bs=100 w/ Rep (90%ID/10%OOD)           0.8883   0.4713   0.9583   0.6319
Random ws=400 bs=100 w/o Rep (90%ID/10%OOD)          0.8871   0.4683   0.9542   0.6283
Random ws=400 bs=100 w/ Rep (80%ID/20%OOD)           0.8892   0.6551   0.9417   0.7726
Random ws=400 bs=100 w/o Rep (80%ID/20%OOD)          0.8988   0.6756   0.9500   0.7896
Random ws=400 bs=100 w/ Rep (70%ID/30%OOD)           0.8976   0.6986   0.9698   0.8121
Random ws=400 bs=100 w/o Rep (70%ID/30%OOD)          0.8985   0.7085   0.9437   0.8093
Random ws=400 bs=100 w/ Rep (60%ID/40%OOD)           0.9029   0.7410   0.9557   0.8348
Random ws=400 bs=100 w/o Rep (60%ID/40%OOD)          0.9107   0.7563   0.9618   0.8468
Random ws=400 bs=100 w/ Rep (50%ID/50%OOD)           0.9134   0.7926   0.9537   0.8658
Random ws=400 bs=100 w/o Rep (50%ID/50%OOD)          0.9175   0.8041   0.9497   0.8708
Random ws=400 bs=100 w/ Rep (40%ID/60%OOD)           0.9094   0.8047   0.9698   0.8796
Random ws=400 bs=100 w/o Rep (40%ID/60%OOD)          0.9128   0.8168   0.9598   0.8825
Random ws=400 bs=100 w/ Rep (30%ID/70%OOD)           0.9154   0.8621   0.9437   0.9011
Random ws=400 bs=100 w/o Rep (30%ID/70%OOD)          0.9236   0.8700   0.9557   0.9108
Random ws=400 bs=100 w/ Rep (20%ID/80%OOD)           0.9263   0.9032   0.9577   0.9297
Random ws=400 bs=100 w/o Rep (20%ID/80%OOD)          0.9243   0.9029   0.9537   0.9276
Random ws=400 bs=100 w/ Rep (10%ID/90%OOD)           0.9254   0.9456   0.9437   0.9446
Random ws=400 bs=100 w/o Rep (10%ID/90%OOD)          0.9281   0.9458   0.9477   0.9467
Random ws=400 bs=100 w/ Rep (5%ID/95%OOD)            0.9465   0.9833   0.9497   0.9662
Random ws=400 bs=100 w/o Rep (5%ID/95%OOD)           0.9352   0.9635   0.9557   0.9596
Random ws=600 bs=20 w/ Rep (95%ID/5%OOD)             0.8846   0.2940   0.9333   0.4471
Random ws=600 bs=20 w/o Rep (95%ID/5%OOD)            0.8796   0.2882   0.9583   0.4432
Random ws=600 bs=20 w/ Rep (90%ID/10%OOD)            0.8842   0.4618   0.9583   0.6233
Random ws=600 bs=20 w/o Rep (90%ID/10%OOD)           0.8892   0.4738   0.9792   0.6386
Random ws=600 bs=20 w/ Rep (80%ID/20%OOD)            0.9033   0.6862   0.9521   0.7976
Random ws=600 bs=20 w/o Rep (80%ID/20%OOD)           0.9008   0.6806   0.9500   0.7930
Random ws=600 bs=20 w/ Rep (70%ID/30%OOD)            0.9058   0.7239   0.9497   0.8216
Random ws=600 bs=20 w/o Rep (70%ID/30%OOD)           0.8980   0.7037   0.9557   0.8106
Random ws=600 bs=20 w/ Rep (60%ID/40%OOD)            0.9091   0.7627   0.9376   0.8412
Random ws=600 bs=20 w/o Rep (60%ID/40%OOD)           0.9128   0.7654   0.9517   0.8484
Random ws=600 bs=20 w/ Rep (50%ID/50%OOD)            0.9246   0.8220   0.9477   0.8804
Random ws=600 bs=20 w/o Rep (50%ID/50%OOD)           0.9081   0.7856   0.9437   0.8574
Random ws=600 bs=20 w/ Rep (40%ID/60%OOD)            0.9135   0.8379   0.9256   0.8795
Random ws=600 bs=20 w/o Rep (40%ID/60%OOD)           0.9197   0.8417   0.9416   0.8889
Random ws=600 bs=20 w/ Rep (30%ID/70%OOD)            0.9252   0.8664   0.9658   0.9134
Random ws=600 bs=20 w/o Rep (30%ID/70%OOD)           0.9178   0.8683   0.9416   0.9035
Random ws=600 bs=20 w/ Rep (20%ID/80%OOD)            0.9284   0.9067   0.9577   0.9315
Random ws=600 bs=20 w/o Rep (20%ID/80%OOD)           0.9263   0.9079   0.9517   0.9293
Random ws=600 bs=20 w/ Rep (10%ID/90%OOD)            0.9294   0.9513   0.9437   0.9475
Random ws=600 bs=20 w/o Rep (10%ID/90%OOD)           0.9227   0.9435   0.9416   0.9426
Random ws=600 bs=20 w/ Rep (5%ID/95%OOD)             0.9368   0.9712   0.9497   0.9603
Random ws=600 bs=20 w/o Rep (5%ID/95%OOD)            0.9335   0.9730   0.9437   0.9581
Random ws=600 bs=50 w/ Rep (95%ID/5%OOD)             0.8812   0.2922   0.9667   0.4487
Random ws=600 bs=50 w/o Rep (95%ID/5%OOD)            0.8812   0.2901   0.9500   0.4444
Random ws=600 bs=50 w/ Rep (90%ID/10%OOD)            0.8792   0.4510   0.9583   0.6133
Random ws=600 bs=50 w/o Rep (90%ID/10%OOD)           0.8892   0.4735   0.9667   0.6356
Random ws=600 bs=50 w/ Rep (80%ID/20%OOD)            0.9050   0.6898   0.9542   0.8007
Random ws=600 bs=50 w/o Rep (80%ID/20%OOD)           0.9050   0.6881   0.9604   0.8017
Random ws=600 bs=50 w/ Rep (70%ID/30%OOD)            0.8980   0.7074   0.9437   0.8086
Random ws=600 bs=50 w/o Rep (70%ID/30%OOD)           0.9008   0.7119   0.9497   0.8138
Random ws=600 bs=50 w/ Rep (60%ID/40%OOD)            0.9138   0.7750   0.9356   0.8478
Random ws=600 bs=50 w/o Rep (60%ID/40%OOD)           0.9035   0.7437   0.9517   0.8350
Random ws=600 bs=50 w/ Rep (50%ID/50%OOD)            0.9157   0.8062   0.9376   0.8670
Random ws=600 bs=50 w/o Rep (50%ID/50%OOD)           0.9128   0.7923   0.9517   0.8647
Random ws=600 bs=50 w/ Rep (40%ID/60%OOD)            0.9224   0.8310   0.9698   0.8951
Random ws=600 bs=50 w/o Rep (40%ID/60%OOD)           0.9080   0.8071   0.9598   0.8768
Random ws=600 bs=50 w/ Rep (30%ID/70%OOD)            0.9203   0.8717   0.9437   0.9063
Random ws=600 bs=50 w/o Rep (30%ID/70%OOD)           0.9145   0.8566   0.9497   0.9008
Random ws=600 bs=50 w/ Rep (20%ID/80%OOD)            0.9284   0.9146   0.9477   0.9308
Random ws=600 bs=50 w/o Rep (20%ID/80%OOD)           0.9232   0.8937   0.9638   0.9274
Random ws=600 bs=50 w/ Rep (10%ID/90%OOD)            0.9294   0.9569   0.9376   0.9472
Random ws=600 bs=50 w/o Rep (10%ID/90%OOD)           0.9308   0.9496   0.9477   0.9486
Random ws=600 bs=50 w/ Rep (5%ID/95%OOD)             0.9595   0.9836   0.9658   0.9746
Random ws=600 bs=50 w/o Rep (5%ID/95%OOD)            0.9384   0.9811   0.9416   0.9610
Random ws=600 bs=100 w/ Rep (95%ID/5%OOD)            0.8792   0.2875   0.9583   0.4423
Random ws=600 bs=100 w/o Rep (95%ID/5%OOD)           0.8792   0.2843   0.9333   0.4358
Random ws=600 bs=100 w/ Rep (90%ID/10%OOD)           0.8958   0.4893   0.9500   0.6459
Random ws=600 bs=100 w/o Rep (90%ID/10%OOD)          0.8833   0.4600   0.9583   0.6216
Random ws=600 bs=100 w/ Rep (80%ID/20%OOD)           0.8996   0.6760   0.9563   0.7921
Random ws=600 bs=100 w/o Rep (80%ID/20%OOD)          0.9008   0.6801   0.9521   0.7934
Random ws=600 bs=100 w/ Rep (70%ID/30%OOD)           0.8907   0.6874   0.9557   0.7997
Random ws=600 bs=100 w/o Rep (70%ID/30%OOD)          0.9035   0.7158   0.9577   0.8193
Random ws=600 bs=100 w/ Rep (60%ID/40%OOD)           0.9035   0.7356   0.9738   0.8381
Random ws=600 bs=100 w/o Rep (60%ID/40%OOD)          0.9112   0.7617   0.9517   0.8462
Random ws=600 bs=100 w/ Rep (50%ID/50%OOD)           0.9093   0.7863   0.9477   0.8595
Random ws=600 bs=100 w/o Rep (50%ID/50%OOD)          0.8998   0.7616   0.9577   0.8485
Random ws=600 bs=100 w/ Rep (40%ID/60%OOD)           0.9190   0.8452   0.9336   0.8872
Random ws=600 bs=100 w/o Rep (40%ID/60%OOD)          0.8964   0.7827   0.9638   0.8638
Random ws=600 bs=100 w/ Rep (30%ID/70%OOD)           0.9121   0.8585   0.9396   0.8972
Random ws=600 bs=100 w/o Rep (30%ID/70%OOD)          0.9088   0.8362   0.9658   0.8964
Random ws=600 bs=100 w/ Rep (20%ID/80%OOD)           0.9181   0.9049   0.9376   0.9209
Random ws=600 bs=100 w/o Rep (20%ID/80%OOD)          0.9161   0.8821   0.9638   0.9212
Random ws=600 bs=100 w/ Rep (10%ID/90%OOD)           0.9430   0.9505   0.9658   0.9581
Random ws=600 bs=100 w/o Rep (10%ID/90%OOD)          0.9254   0.9438   0.9457   0.9447
Random ws=600 bs=100 w/ Rep (5%ID/95%OOD)            0.9498   0.9814   0.9557   0.9684
Random ws=600 bs=100 w/o Rep (5%ID/95%OOD)           0.9449   0.9813   0.9497   0.9652
Wave ws=200 bs=20 Standard Block25                   0.9358   0.9588   0.9556   0.9572
Wave ws=200 bs=20 Standard Block50                   0.9346   0.9607   0.9517   0.9562
Wave ws=200 bs=20 Standard Block100                  0.9271   0.9656   0.9361   0.9506
Wave ws=200 bs=20 Standard Block200                  0.9025   0.9644   0.9033   0.9329
Wave ws=200 bs=20 Standard Block500                  0.8708   0.9064   0.8847   0.8954
Wave ws=200 bs=20 FastSwitch Block25                 0.9313   0.9394   0.9587   0.9490
Wave ws=200 bs=20 FastSwitch Block50                 0.9350   0.9365   0.9681   0.9521
Wave ws=200 bs=20 FastSwitch Block100                0.9321   0.9394   0.9600   0.9496
Wave ws=200 bs=20 FastSwitch Block200                0.9129   0.9371   0.9319   0.9345
Wave ws=200 bs=20 FastSwitch Block500                0.8504   0.9032   0.8520   0.8768
Wave ws=200 bs=20 Binary Block25                     0.9125   0.8873   0.9450   0.9153
Wave ws=200 bs=20 Binary Block50                     0.9187   0.8917   0.9533   0.9215
Wave ws=200 bs=20 Binary Block100                    0.9179   0.8933   0.9492   0.9204
Wave ws=200 bs=20 Binary Block200                    0.9096   0.8693   0.9642   0.9143
Wave ws=200 bs=20 Binary Block500                    0.9008   0.8451   0.9330   0.8869
Wave ws=200 bs=50 Standard Block25                   0.9396   0.9620   0.9572   0.9596
Wave ws=200 bs=50 Standard Block50                   0.9346   0.9556   0.9572   0.9564
Wave ws=200 bs=50 Standard Block100                  0.9229   0.9681   0.9278   0.9475
Wave ws=200 bs=50 Standard Block200                  0.9267   0.9688   0.9322   0.9502
Wave ws=200 bs=50 Standard Block500                  0.8538   0.8981   0.8640   0.8807
Wave ws=200 bs=50 FastSwitch Block25                 0.9313   0.9383   0.9600   0.9490
Wave ws=200 bs=50 FastSwitch Block50                 0.9317   0.9383   0.9606   0.9494
Wave ws=200 bs=50 FastSwitch Block100                0.9350   0.9440   0.9594   0.9516
Wave ws=200 bs=50 FastSwitch Block200                0.9413   0.9489   0.9637   0.9563
Wave ws=200 bs=50 FastSwitch Block500                0.8521   0.8978   0.8613   0.8792
Wave ws=200 bs=50 Binary Block25                     0.9137   0.8828   0.9542   0.9171
Wave ws=200 bs=50 Binary Block50                     0.9054   0.8650   0.9608   0.9104
Wave ws=200 bs=50 Binary Block100                    0.9046   0.8578   0.9700   0.9104
Wave ws=200 bs=50 Binary Block200                    0.8996   0.8418   0.9842   0.9074
Wave ws=200 bs=50 Binary Block500                    0.9058   0.8407   0.9550   0.8942
Wave ws=200 bs=100 Standard Block25                  0.9371   0.9513   0.9656   0.9584
Wave ws=200 bs=100 Standard Block50                  0.9358   0.9507   0.9644   0.9575
Wave ws=200 bs=100 Standard Block100                 0.9329   0.9638   0.9461   0.9549
Wave ws=200 bs=100 Standard Block200                 0.9413   0.9631   0.9583   0.9607
Wave ws=200 bs=100 Standard Block500                 0.8712   0.9016   0.8913   0.8964
Wave ws=200 bs=100 FastSwitch Block25                0.9313   0.9383   0.9600   0.9490
Wave ws=200 bs=100 FastSwitch Block50                0.9300   0.9366   0.9600   0.9481
Wave ws=200 bs=100 FastSwitch Block100               0.9367   0.9361   0.9712   0.9534
Wave ws=200 bs=100 FastSwitch Block200               0.9371   0.9362   0.9719   0.9537
Wave ws=200 bs=100 FastSwitch Block500               0.8712   0.9016   0.8913   0.8964
Wave ws=200 bs=100 Binary Block25                    0.9113   0.8782   0.9550   0.9150
Wave ws=200 bs=100 Binary Block50                    0.9042   0.8598   0.9658   0.9097
Wave ws=200 bs=100 Binary Block100                   0.9050   0.8547   0.9758   0.9113
Wave ws=200 bs=100 Binary Block200                   0.9029   0.8481   0.9817   0.9100
Wave ws=200 bs=100 Binary Block500                   0.9025   0.8389   0.9480   0.8901
Wave ws=400 bs=20 Standard Block25                   0.9400   0.9652   0.9544   0.9598
Wave ws=400 bs=20 Standard Block50                   0.9350   0.9639   0.9489   0.9563
Wave ws=400 bs=20 Standard Block100                  0.9308   0.9632   0.9439   0.9534
Wave ws=400 bs=20 Standard Block200                  0.9279   0.9694   0.9333   0.9510
Wave ws=400 bs=20 Standard Block500                  0.8696   0.9147   0.8727   0.8932
Wave ws=400 bs=20 FastSwitch Block25                 0.9342   0.9434   0.9587   0.9510
Wave ws=400 bs=20 FastSwitch Block50                 0.9379   0.9443   0.9637   0.9539
Wave ws=400 bs=20 FastSwitch Block100                0.9325   0.9416   0.9581   0.9498
Wave ws=400 bs=20 FastSwitch Block200                0.9275   0.9456   0.9456   0.9456
Wave ws=400 bs=20 FastSwitch Block500                0.8721   0.9157   0.8760   0.8954
Wave ws=400 bs=20 Binary Block25                     0.9150   0.8837   0.9558   0.9183
Wave ws=400 bs=20 Binary Block50                     0.9179   0.8933   0.9492   0.9204
Wave ws=400 bs=20 Binary Block100                    0.9163   0.8930   0.9458   0.9187
Wave ws=400 bs=20 Binary Block200                    0.9163   0.8949   0.9433   0.9185
Wave ws=400 bs=20 Binary Block500                    0.9183   0.8545   0.9690   0.9082
Wave ws=400 bs=50 Standard Block25                   0.9371   0.9629   0.9528   0.9578
Wave ws=400 bs=50 Standard Block50                   0.9342   0.9607   0.9511   0.9559
Wave ws=400 bs=50 Standard Block100                  0.9350   0.9577   0.9556   0.9566
Wave ws=400 bs=50 Standard Block200                  0.9200   0.9664   0.9256   0.9455
Wave ws=400 bs=50 Standard Block500                  0.8804   0.9101   0.8973   0.9037
Wave ws=400 bs=50 FastSwitch Block25                 0.9333   0.9390   0.9625   0.9506
Wave ws=400 bs=50 FastSwitch Block50                 0.9325   0.9384   0.9619   0.9500
Wave ws=400 bs=50 FastSwitch Block100                0.9296   0.9182   0.9819   0.9490
Wave ws=400 bs=50 FastSwitch Block200                0.9321   0.9510   0.9469   0.9490
Wave ws=400 bs=50 FastSwitch Block500                0.8700   0.9057   0.8840   0.8947
Wave ws=400 bs=50 Binary Block25                     0.9175   0.8920   0.9500   0.9201
Wave ws=400 bs=50 Binary Block50                     0.9154   0.8891   0.9492   0.9182
Wave ws=400 bs=50 Binary Block100                    0.9150   0.8903   0.9467   0.9176
Wave ws=400 bs=50 Binary Block200                    0.9175   0.8945   0.9467   0.9198
Wave ws=400 bs=50 Binary Block500                    0.9125   0.8405   0.9750   0.9028
Wave ws=400 bs=100 Standard Block25                  0.9321   0.9601   0.9489   0.9545
Wave ws=400 bs=100 Standard Block50                  0.9329   0.9607   0.9494   0.9550
Wave ws=400 bs=100 Standard Block100                 0.9387   0.9460   0.9739   0.9598
Wave ws=400 bs=100 Standard Block200                 0.9192   0.9674   0.9233   0.9449
Wave ws=400 bs=100 Standard Block500                 0.9008   0.9207   0.9207   0.9207
Wave ws=400 bs=100 FastSwitch Block25                0.9313   0.9405   0.9575   0.9489
Wave ws=400 bs=100 FastSwitch Block50                0.9296   0.9392   0.9563   0.9477
Wave ws=400 bs=100 FastSwitch Block100               0.9346   0.9381   0.9656   0.9516
Wave ws=400 bs=100 FastSwitch Block200               0.9350   0.9408   0.9631   0.9518
Wave ws=400 bs=100 FastSwitch Block500               0.8858   0.9142   0.9020   0.9081
Wave ws=400 bs=100 Binary Block25                    0.9196   0.8943   0.9517   0.9221
Wave ws=400 bs=100 Binary Block50                    0.9192   0.8936   0.9517   0.9217
Wave ws=400 bs=100 Binary Block100                   0.9196   0.8949   0.9508   0.9220
Wave ws=400 bs=100 Binary Block200                   0.9050   0.8589   0.9692   0.9107
Wave ws=400 bs=100 Binary Block500                   0.9154   0.8426   0.9800   0.9061
Wave ws=600 bs=20 Standard Block25                   0.9408   0.9652   0.9556   0.9604
Wave ws=600 bs=20 Standard Block50                   0.9367   0.9645   0.9506   0.9575
Wave ws=600 bs=20 Standard Block100                  0.9317   0.9627   0.9456   0.9540
Wave ws=600 bs=20 Standard Block200                  0.9342   0.9633   0.9483   0.9558
Wave ws=600 bs=20 Standard Block500                  0.8871   0.9212   0.8960   0.9084
Wave ws=600 bs=20 FastSwitch Block25                 0.9333   0.9423   0.9587   0.9504
Wave ws=600 bs=20 FastSwitch Block50                 0.9375   0.9448   0.9625   0.9536
Wave ws=600 bs=20 FastSwitch Block100                0.9321   0.9416   0.9575   0.9495
Wave ws=600 bs=20 FastSwitch Block200                0.9233   0.9291   0.9581   0.9434
Wave ws=600 bs=20 FastSwitch Block500                0.8792   0.9196   0.8840   0.9014
Wave ws=600 bs=20 Binary Block25                     0.9196   0.8931   0.9533   0.9222
Wave ws=600 bs=20 Binary Block50                     0.9200   0.8956   0.9508   0.9224
Wave ws=600 bs=20 Binary Block100                    0.9150   0.8915   0.9450   0.9175
Wave ws=600 bs=20 Binary Block200                    0.9117   0.8908   0.9383   0.9140
Wave ws=600 bs=20 Binary Block500                    0.9225   0.8551   0.9800   0.9133
Wave ws=600 bs=50 Standard Block25                   0.9392   0.9625   0.9561   0.9593
Wave ws=600 bs=50 Standard Block50                   0.9354   0.9598   0.9539   0.9568
Wave ws=600 bs=50 Standard Block100                  0.9308   0.9574   0.9500   0.9537
Wave ws=600 bs=50 Standard Block200                  0.9279   0.9635   0.9394   0.9513
Wave ws=600 bs=50 Standard Block500                  0.8917   0.9095   0.9180   0.9137
Wave ws=600 bs=50 FastSwitch Block25                 0.9350   0.9397   0.9644   0.9519
Wave ws=600 bs=50 FastSwitch Block50                 0.9329   0.9379   0.9631   0.9504
Wave ws=600 bs=50 FastSwitch Block100                0.9304   0.9398   0.9569   0.9483
Wave ws=600 bs=50 FastSwitch Block200                0.9304   0.9420   0.9544   0.9482
Wave ws=600 bs=50 FastSwitch Block500                0.8883   0.9079   0.9140   0.9110
Wave ws=600 bs=50 Binary Block25                     0.9179   0.8952   0.9467   0.9202
Wave ws=600 bs=50 Binary Block50                     0.9158   0.8905   0.9483   0.9185
Wave ws=600 bs=50 Binary Block100                    0.9167   0.8925   0.9475   0.9192
Wave ws=600 bs=50 Binary Block200                    0.9167   0.8925   0.9475   0.9192
Wave ws=600 bs=50 Binary Block500                    0.9192   0.8427   0.9910   0.9108
Wave ws=600 bs=100 Standard Block25                  0.9329   0.9586   0.9517   0.9551
Wave ws=600 bs=100 Standard Block50                  0.9350   0.9592   0.9539   0.9565
Wave ws=600 bs=100 Standard Block100                 0.9371   0.9619   0.9539   0.9579
Wave ws=600 bs=100 Standard Block200                 0.9383   0.9494   0.9694   0.9593
Wave ws=600 bs=100 Standard Block500                 0.9025   0.9209   0.9233   0.9221
Wave ws=600 bs=100 FastSwitch Block25                0.9308   0.9410   0.9563   0.9485
Wave ws=600 bs=100 FastSwitch Block50                0.9300   0.9393   0.9569   0.9480
Wave ws=600 bs=100 FastSwitch Block100               0.9358   0.9419   0.9631   0.9524
Wave ws=600 bs=100 FastSwitch Block200               0.9346   0.9257   0.9806   0.9524
Wave ws=600 bs=100 FastSwitch Block500               0.8958   0.9156   0.9180   0.9168
Wave ws=600 bs=100 Binary Block25                    0.9146   0.8890   0.9475   0.9173
Wave ws=600 bs=100 Binary Block50                    0.9154   0.8891   0.9492   0.9182
Wave ws=600 bs=100 Binary Block100                   0.9050   0.8558   0.9742   0.9111
Wave ws=600 bs=100 Binary Block200                   0.8996   0.8513   0.9683   0.9060
Wave ws=600 bs=100 Binary Block500                   0.9225   0.8479   0.9920   0.9143

--- Random Streams (all configurations) ---
Acc: 0.9107 ± 0.0180
Prec: 0.7469 ± 0.1976
Rec:  0.9523 ± 0.0112
F1:   0.8198 ± 0.1471

--- Wave Streams (all configurations) ---
Acc: 0.9186 ± 0.0201
Prec: 0.9196 ± 0.0379
Rec:  0.9473 ± 0.0263
F1:   0.9326 ± 0.0227

Config ws=200, bs=20:  Acc=0.9116±0.0207, F1=0.8630±0.1278 )
Config ws=200, bs=50:  Acc=0.9118±0.0220, F1=0.8641±0.1237 )
Config ws=200, bs=100:  Acc=0.9120±0.0201, F1=0.8634±0.1279 )
Config ws=400, bs=20:  Acc=0.9145±0.0183, F1=0.8669±0.1250 )
Config ws=400, bs=50:  Acc=0.9154±0.0191, F1=0.8676±0.1255 )
Config ws=400, bs=100:  Acc=0.9142±0.0177, F1=0.8654±0.1287 )
Config ws=600, bs=20:  Acc=0.9156±0.0175, F1=0.8670±0.1275 )
Config ws=600, bs=50:  Acc=0.9159±0.0182, F1=0.8671±0.1282 )
Config ws=600, bs=100:  Acc=0.9140±0.0186, F1=0.8649±0.1288 )

OVERALL AVERAGE PERFORMANCE (over all tests)
Mean Accuracy: 0.9139 ± 0.0193
Mean Precision (OOD): 0.8169 ± 0.1760
Mean Recall (OOD): 0.9503 ± 0.0190
Mean F1-score (OOD): 0.8655 ± 0.1270
"""

# STAGE 5

Find out what happen to the model and distance distribution a system retrain.

Now the model has been retrain wiht also the OOD data samples

In [ ]:
REPO_ID = "tomasconti/Drift_Detection"
print("Downloading native 512-D embeddings from HuggingFace")

path_train = hf_hub_download(repo_id=REPO_ID, filename="84C/train_embeddings_84C.npy")
path_val = hf_hub_download(repo_id=REPO_ID, filename="84C/val_embeddings_84C.npy")
path_test = hf_hub_download(repo_id=REPO_ID, filename="84C/test_embeddings_84C.npy")


test_embeddings_84C = np.load(path_test)
val_embeddings_84C = np.load(path_val)
train_embeddings_84C = np.load(path_train)


In [ ]:
train_labels = train_generator.classes
tsne = TSNE( n_components=2, perplexity=50, learning_rate="auto", init="pca", random_state=42 )
embeddings_2d = tsne.fit_transform(train_embeddings_84C)
print("t-SNE shape:", embeddings_2d.shape)
plt.figure(figsize=(12,10))
scatter = plt.scatter( embeddings_2d[:,0], embeddings_2d[:,1], )
plt.title("t-SNE visualization of test set embeddings")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.show()

Are visible some new clusters into the embedding (ODD new clusters)

In [ ]:
results = []
for linkage in ['ward', 'complete', 'average', 'single']:
    for k in range(75, 91):
        model = AgglomerativeClustering( n_clusters=k,linkage=linkage)
        labels = model.fit_predict(train_embeddings_84C)
        results.append({ "Linkage": linkage,"K": k,"Silhouette": silhouette_score(train_embeddings_84C, labels)})
df_results = pd.DataFrame(results)
plt.figure(figsize=(10,5))
sns.lineplot( data=df_results, x="K", y="Silhouette", hue="Linkage", marker="o")
plt.title("Agglomerative Clustering of embeddings")
plt.xlabel("Clusters")
plt.ylabel("Silhouette Score")
plt.grid()
plt.legend()
plt.show()

In [ ]:
n_clusters = 84
cluster_model = AgglomerativeClustering( n_clusters=n_clusters, linkage='ward')
labels = cluster_model.fit_predict(train_embeddings_84C)
centroids = np.array([ train_embeddings_84C[labels == i].mean(axis=0) for i in range(n_clusters)])
print("Centroids shape:", centroids.shape)

In [ ]:
all_data = np.concatenate( [train_embeddings_84C, centroids], axis=0 )
all_2d = TSNE( n_components=2, perplexity=50, random_state=42, init="pca").fit_transform(all_data)

train_2d = all_2d[:len(train_embeddings_84C)]
centroids_2d = all_2d[len(train_embeddings_84C):]

plt.figure(figsize=(12,8))
plt.scatter( train_2d[:,0], train_2d[:,1], s=15, alpha=0.4, label="Train embeddings" )

plt.scatter( centroids_2d[:,0], centroids_2d[:,1], marker="X", s=120, c="red", edgecolors="black", label="Cluster centroids" )

plt.title("t-SNE: Train embeddings + Agglomerative Centroids")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.legend()
plt.grid(True)

plt.show()

OOD clusters are now part of the system's semantic embeddings, making it possible to define new semantic anchor centroids

In [ ]:
knn = NearestNeighbors(n_neighbors=1, metric="euclidean").fit(centroids)
data_embeddings = { "Train": train_embeddings_84C,"Val": val_embeddings_84C, "Test": test_embeddings_84C }
distances = {}
for name, emb in data_embeddings.items():
    flat_distances = knn.kneighbors(emb)[0].flatten()
    distances[name] = flat_distances

plt.figure(figsize=(10,6))

for name, dist in distances.items():
    sns.kdeplot(dist, fill=True, alpha=0.25, label=name)
    plt.axvline(np.median(dist), linestyle="--", label=f"{name} median")

plt.title("Distance to nearest centroid (K=1)")
plt.xlabel("Euclidean distance")
plt.ylabel("Density")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

rows = []
for name, dist_list in distances.items():
    for d in dist_list:
        row_dict = { "Dataset": name, "Distance": d }
        rows.append(row_dict)

df_dist = pd.DataFrame(rows)

plt.figure(figsize=(10,5))
sns.boxplot(data=df_dist, x="Dataset", y="Distance") #required a dataframe
plt.title("Distance distribution to nearest centroid")
plt.grid(axis="y", alpha=0.3)
plt.show()

After retraining the ResNet CNN with the OOD samples, the model is now able to recognize the new clusters. New anchor centroids have been introduced for the distance calculation, which helps bring the test distance distribution into a stable state. We no longer see the old, two Gaussian distribution separating ID and OOD data. Thanks to the retraining, the system can now target the old OOD data, which has officially become part of the In-Distribution (ID) data